# Looking at the data before you trust it

**Module 1 · Session 03, part 1**

Last week we decided what one row means and built a table. We never looked at the shape of a
single column. This notebook does that, on the same file, and it needs no joins: every audio
measurement is already sitting on every row.

Nobody here is being trained as a statistician, and you will not be asked to derive anything.
What you need is narrower and harder to hand off: ask a question worth asking, look at what
comes back, and judge whether a pattern is worth anyone's money.

## The five questions

Work in this order on any table, not just this one. The notebook goes through them once.

| | Question | Where to look |
|---|---|---|
| 1 | How big is it? | `.shape`, `.info()` |
| 2 | What is missing? | `.isna().sum()` |
| 3 | Is anything impossible? | `.describe()`, and read `min` and `max` first |
| 4 | What groups are there, and how big? | `.value_counts()` |
| 5 | What moves with what? | `.corr()`, and always a picture |

None of that is difficult and all of it is skippable, which is exactly why it gets skipped.


In [ ]:
# Setup. Same file as last week. Nothing else to install.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/aaubs/ds-master/codex/m1-pandas-2026/data/M1_2026/spotify_songs.csv"
local = [Path("data/M1_2026/spotify_songs.csv"), Path("../data/M1_2026/spotify_songs.csv"),
         Path("ds-master/data/M1_2026/spotify_songs.csv"), Path("../ds-master/data/M1_2026/spotify_songs.csv")]

path = next((p for p in local if p.exists()), None)
songs = pd.read_csv(path if path is not None else URL)
songs = songs.rename(columns={"track_name": "title", "track_artist": "artist",
                             "track_popularity": "popularity", "playlist_genre": "genre"})

# One row = one track on one playlist, which is the grain we settled on last week.
# One row per song instead:
tracks = songs.drop_duplicates("track_id")

print(f"Rows (a track on a playlist): {len(songs):,}")
print(f"Distinct songs:               {len(tracks):,}")


Two frames, one file. `songs` has a row for every time a track appears on a playlist, so a
popular song is in there several times. `tracks` has one row per song.

Which one you use depends on the question. For "what is a typical song", use `tracks`. For
"what is on these playlists", use `songs`. Getting this wrong is not a coding mistake, it is
an answer to a different question.

## 1 and 2. How big, and what is missing


In [ ]:
print("Rows and columns:", songs.shape)

# Missing values, in the columns we are about to use.
display(songs[["title", "artist", "energy", "danceability", "tempo"]].isna().sum().to_frame("missing"))


Five missing titles and artists, no missing audio measurements. Session 02 covered what to
do about that: nothing here, because the analysis is about energy and the rows with an
unknown artist still carry it.

## 3. Is anything impossible?


In [ ]:
audio = ["energy", "danceability", "valence", "tempo", "duration_ms"]
display(tracks[audio].describe().round(2))


Read the row labelled `min` before anything else. A duration of 4,000 milliseconds is four
seconds, and a tempo of 0 beats per minute is not a slow song. We come back to those.

Now the part `describe` cannot show you. Every column above has a mean. Only some of them
have a mean worth reporting.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, column in zip(axes.flat, ["energy", "danceability", "valence", "tempo"]):
    tracks[column].plot(kind="hist", bins=40, ax=ax, color="#2f6f9f", edgecolor="white")
    ax.axvline(tracks[column].mean(), color="#d97941", linewidth=2)   # the mean, in orange
    ax.set(title=column, ylabel="songs")
plt.tight_layout()
plt.show()


`valence` is spread almost evenly from 0 to 1. Its mean is 0.51, which is a real number
describing nothing in particular: there is no cluster of songs there.

`tempo` has a spike just above 120 beats per minute and a second bump near 100, because
produced music is written to a small number of conventional tempos. The mean sits between
them.

`energy` leans towards the top of the range. `danceability` is the only one here with a
single clear hump, and it is the only one whose mean is a fair summary.

None of that is visible in `describe`. It took one plot.


> **Judgement call.** An agent asked to "compare mean energy across genres" will produce correct means and never
mention that two of these columns have no middle. The histogram is ten seconds of work and it
is not in the request. Knowing to ask for it is the part that is yours.


### Look at the row, not the number

The minimum duration was four seconds. Look at the row rather than deciding in the abstract.


In [ ]:
display(tracks.nsmallest(3, "duration_ms")[["title", "artist", "duration_ms", "tempo", "energy"]])


In [ ]:
# A separate check, on a different column.
display(tracks.loc[tracks["tempo"] == 0, ["title", "artist", "duration_ms", "tempo"]])


It is the same row both times. A four-second track with a tempo of zero is not a short song
with no beat, it is a record that failed to be a song.

Nobody suspected that row. It turned up because we looked at the minimum of two unrelated
columns. That is what exploratory work is for.


In [ ]:
print("Songs under one minute:", int((tracks["duration_ms"] < 60_000).sum()))
print("Songs with tempo exactly 0:", int((tracks["tempo"] == 0).sum()))
print("Out of:", len(tracks))


Twenty-five songs under a minute out of 28,356. So there is no crowd of broken rows here,
just a handful, and they cannot move a mean.

The defensible decision for this file is to leave them in and say that you looked. If you
were quoting a typical song length in a contract, you would drop them and say so. The
decision changes with the claim, which is why nobody can write it down for you in advance.

## 4. What groups are there, and how big?

The least interesting cell in any analysis, and the one that prevents the most mistakes.


In [ ]:
display(songs["genre"].value_counts().to_frame("rows"))


Between about 4,900 and 6,000 rows per genre, which is unusually balanced. When it is not,
a difference between a group of 40 and a group of 40,000 tells you more about sample sizes
than about the world.

Print these counts next to every summary you produce. It is a habit, not a technique.

## 5. What moves with what?

A correlation is one number for how two columns move together. Easy to compute, easy to
over-read, so always look at the picture too.


In [ ]:
correlations = tracks[["energy", "danceability", "loudness", "valence", "acousticness", "popularity"]].corr()
display(correlations["energy"].drop("energy").sort_values(ascending=False).round(3).to_frame("with energy"))


Energy and loudness sit at 0.68. That is high, and close to a tautology: both partly measure
how much is going on in the recording.

Energy and danceability sit at -0.08, which is nothing. Worth saying out loud, because
"energetic" and "danceable" sound like they belong together in English and the data disagrees.

Deciding which of these numbers is a finding is not in the table.


In [ ]:
# 28,000 points would be a solid block of ink, so plot a reproducible sample.
sample = tracks.sample(2_000, random_state=2026)
ax = sample.plot(kind="scatter", x="loudness", y="energy", alpha=0.25, s=12,
                 figsize=(8, 5), color="#2f6f9f")
ax.set(title="Energy against loudness, 2,000 sampled songs", xlabel="loudness (dB)", ylabel="energy")
plt.show()


Two things the number 0.68 did not tell you. The cloud is wide, so loudness does not predict
energy for any individual song. And there is a tail of very quiet tracks to the left, which is
a kind of music rather than a measurement error.

## Practice

Pick one numeric column we have not looked at, plot it, and say in one sentence whether its
mean is a fair summary. Then find the most extreme row in that column and decide whether it
belongs in the file.


In [ ]:
# Your turn. One column, one plot, one sentence, one decision.


## Solution, using `speechiness`


In [ ]:
ax = tracks["speechiness"].plot(kind="hist", bins=50, figsize=(9, 4),
                                color="#2f6f9f", edgecolor="white")
ax.axvline(tracks["speechiness"].mean(), color="#d97941", linewidth=2)
ax.set(title="speechiness", xlabel="speechiness", ylabel="songs")
plt.show()

print("mean:  ", round(tracks["speechiness"].mean(), 3))
print("median:", round(tracks["speechiness"].median(), 3))
display(tracks.nlargest(3, "speechiness")[["title", "artist", "speechiness", "duration_ms"]])


## Next

That was the five questions, once, on one file. The order is the point: you cannot judge a
correlation until you know how big the groups are, and you cannot trust a mean until you have
seen the shape and checked the extremes.

Part 2 takes the obvious follow-up. Genres clearly differ in energy, but how much of that is a
real difference and how much is thirty thousand rows making everything look certain?
